# 🧠 Neural Networks & Deep Learning
**ISLP Chapter 10 · Pattern Recognition for the Rest of Us**

> Neural networks stack layers of simple computations to learn arbitrarily complex patterns. They power image recognition, language models, and game-playing AI. This notebook covers the fundamentals that apply to all deep learning.

### What you'll learn
- Architecture: layers, neurons, weights, activations
- Forward pass: how predictions are made
- Backpropagation: how weights are updated
- Key activations: ReLU, sigmoid, softmax
- Regularization: dropout, batch normalization
- Practical deep learning with Keras/TensorFlow

### Dataset: Digits (sklearn) + MNIST Fashion (via Keras)
### Time: ~75 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('✓ Setup complete')
# TensorFlow/Keras is pre-installed in Colab
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, classification_report
import warnings; warnings.filterwarnings('ignore')

print(f'✓ TensorFlow version: {tf.__version__}')
print(f'✓ GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 🎯 Part 1 — Architecture: Layers, Neurons, Activations

A feedforward neural network is a series of **layers**:

```
Input → [Hidden Layer 1] → [Hidden Layer 2] → ... → Output
```

Each **neuron** computes:
```
z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b   (linear combination)
a = f(z)                              (activation function)
```

**Common activation functions:**
- **ReLU:** f(z) = max(0, z) — default for hidden layers
- **Sigmoid:** f(z) = 1/(1+e^-z) — output for binary classification
- **Softmax:** converts logits to probabilities — output for multiclass
- **Linear:** no activation — output for regression

In [ ]:
# Visualize activation functions
z = np.linspace(-4, 4, 200)
activations = {
    'ReLU': np.maximum(0, z),
    'Sigmoid': 1/(1+np.exp(-z)),
    'Tanh': np.tanh(z),
    'Leaky ReLU': np.where(z>0, z, 0.1*z)
}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
colors_a = ['#1e5fa8','#e85d20','#1a7a45','#6b46c1']
for ax, (name, vals), color in zip(axes, activations.items(), colors_a):
    ax.plot(z, vals, color=color, lw=2.5)
    ax.axhline(0, color='black', lw=0.5, alpha=0.3)
    ax.axvline(0, color='black', lw=0.5, alpha=0.3)
    ax.set_title(f'{name}')
    ax.set_xlabel('z'); ax.set_ylim(-1.5, 1.5)
plt.suptitle('Activation Functions — Applied After Each Linear Layer', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print('📌 ReLU is the default for hidden layers: simple, fast, avoids vanishing gradients')
print('   Sigmoid/Softmax go on the OUTPUT layer only (for probability outputs)')

In [ ]:
# Build and train a network with sklearn MLPClassifier — simpler API
digits = load_digits()
X_d = digits.data; y_d = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.2, random_state=42, stratify=y_d)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)

# Compare different architectures
architectures = [
    ((32,), 'Shallow (1 layer, 32 neurons)'),
    ((64, 32), 'Medium (2 layers)'),
    ((128, 64, 32), 'Deep (3 layers)'),
    ((256, 128, 64, 32), 'Deeper (4 layers)'),
]
print(f'Digit classification — {len(digits.target_names)} classes, {X_d.shape[1]} features\n')
print(f'{"Architecture":<35} {"Test Acc":>10} {"Train Acc":>12}')
print('-'*60)
for arch, name in architectures:
    mlp = MLPClassifier(hidden_layer_sizes=arch, max_iter=500, random_state=42,
                       early_stopping=True, validation_fraction=0.1)
    mlp.fit(X_tr_s, y_tr)
    train_acc = mlp.score(X_tr_s, y_tr)
    test_acc  = mlp.score(X_te_s, y_te)
    print(f'{name:<35} {test_acc:>10.4f} {train_acc:>12.4f}')

In [ ]:
# Training curves: loss over epochs
mlp_best = MLPClassifier(hidden_layer_sizes=(128,64), max_iter=300, random_state=42,
                         early_stopping=True, validation_fraction=0.15,
                         verbose=False, warm_start=False)
mlp_best.fit(X_tr_s, y_tr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(mlp_best.loss_curve_, color='#e85d20', lw=2, label='Training loss')
if hasattr(mlp_best, 'validation_scores_'):
    axes[0].plot(mlp_best.validation_scores_, color='#1e5fa8', lw=2, label='Validation score')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss / Score')
axes[0].set_title('Training Curves')
axes[0].legend()

# Confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
cm = confusion_matrix(y_te, mlp_best.predict(X_te_s))
ConfusionMatrixDisplay(cm, display_labels=digits.target_names).plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Confusion Matrix — MLP (acc={mlp_best.score(X_te_s,y_te):.3f})')
plt.tight_layout(); plt.show()

## 🔥 Part 2 — Deep Learning with Keras: Fashion MNIST

Keras is TensorFlow's high-level API. It makes building neural networks as simple as stacking LEGO bricks.

In [ ]:
# Load Fashion MNIST — 60k images of clothing items
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = ['T-shirt','Trouser','Pullover','Dress','Coat','Sandal','Shirt','Sneaker','Bag','Ankle boot']

print(f'Training set: {X_train.shape}, Test set: {X_test.shape}')
print(f'Classes: {class_names}')

# Preview some images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flatten(), np.random.choice(len(X_train), 10, replace=False)):
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(class_names[y_train[idx]], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion MNIST — Sample Images', fontsize=12)
plt.tight_layout(); plt.show()

# Normalize
X_train_n = X_train / 255.0
X_test_n  = X_test  / 255.0

In [ ]:
# Build and train the model
tf.random.set_seed(42)
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),          # 784 inputs
    layers.Dense(128, activation='relu'),           # Hidden layer 1
    layers.Dropout(0.3),                            # Regularization
    layers.Dense(64, activation='relu'),            # Hidden layer 2
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')          # 10 class probabilities
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_train_n, y_train, epochs=15, batch_size=128,
                   validation_split=0.1, verbose=1)

In [ ]:
# Plot training curves and evaluate
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], color='#e85d20', label='Train loss')
axes[0].plot(history.history['val_loss'], color='#1e5fa8', label='Val loss')
axes[0].set_title('Loss Curve'); axes[0].legend()
axes[1].plot(history.history['accuracy'], color='#e85d20', label='Train acc')
axes[1].plot(history.history['val_accuracy'], color='#1e5fa8', label='Val acc')
axes[1].set_title('Accuracy Curve'); axes[1].legend()
plt.tight_layout(); plt.show()

test_loss, test_acc = model.evaluate(X_test_n, y_test, verbose=0)
print(f'\nTest accuracy: {test_acc:.4f}')
print(classification_report(y_test, model.predict(X_test_n).argmax(axis=1), target_names=class_names))

In [ ]:
# Exercise workspace
# Task 1: Add batch normalization layers — does accuracy improve?
# from tensorflow.keras.layers import BatchNormalization
# YOUR CODE HERE

# Task 2: Try different learning rates (0.001, 0.01, 0.1)
# optimizer = keras.optimizers.Adam(learning_rate=0.001)
# YOUR CODE HERE

# Task 3: Visualize the weights of the first hidden layer as images
# weights = model.layers[1].get_weights()[0].reshape(28,28,-1)  # 128 filters
# YOUR CODE HERE

In [ ]:
answers = {
    'q1': '',  # What does ReLU do? Why is it preferred over sigmoid in hidden layers?
    'q2': '',  # What is dropout and how does it prevent overfitting?
    'q3': '',  # What does backpropagation compute?
    'q4': '',  # What is the purpose of the softmax activation on the output layer?
    'q5': '',  # What does validation loss increasing while training loss decreases indicate?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')

## 📚 Further Reading
- [ISLP Ch 10](https://intro-stat-learning.github.io/ISLP/) — Neural Networks
- [Keras tutorials](https://keras.io/examples/)
- [3Blue1Brown: Neural Networks playlist](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)
- [Model Explainability notebook](model-explainability.ipynb) — SHAP for neural networks

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*